# Model 1: Network Intrusion Detection (+ Duration Regression)

**Project:** AI-Based Cybersecurity Threat Detection System

**Part 1 (classification):** cleans the network-traffic dataset, trains and evaluates models to classify network activity, selects the 10 most important features, and saves the final model for the web app.

**Part 2 (regression):** trains a Linear Regression model that predicts how long a network connection lasts, reports R², and saves it for the web app.



## Step 1: Connect Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Import libraries

In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    r2_score
)

## Step 3: Load the dataset

In [3]:
DATA_PATH = '/content/drive/MyDrive/ML_Project/wednesday.csv'
OUTPUT_FOLDER = '/content/drive/MyDrive/ML_Project/'

df = pd.read_csv(DATA_PATH)

# Remove accidental spaces from column names
df.columns = df.columns.str.strip()

print('Dataset shape:', df.shape)
display(df.head())
print('Target labels before cleaning:')
print(df['Label'].value_counts())

Dataset shape: (496640, 89)


,Src IP dec,Src Port,Dst IP dec,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label,Attempted Category
0,3232238089,13793,885387958,443,6,43:34.8,187889,1,1,0,...,0,0.0,0.0,0,0,-1,-1,187889,BENIGN,-1
1,3232238089,13793,885387958,443,6,43:35.5,187758,1,1,0,...,0,0.0,0.0,0,0,-1,-1,187758,BENIGN,-1
2,3232238089,13794,910236785,443,6,43:36.4,189882,1,1,0,...,0,0.0,0.0,0,0,-1,-1,189882,BENIGN,-1
3,3232238089,13794,910236785,443,6,43:37.1,190117,1,1,0,...,0,0.0,0.0,0,0,-1,-1,190117,BENIGN,-1
4,3232238089,13796,885387958,443,6,43:38.0,188603,1,1,0,...,0,0.0,0.0,0,0,-1,-1,188603,BENIGN,-1


Target labels before cleaning:
Label
BENIGN                          319119
DoS Hulk                        158468
DoS GoldenEye                     7567
DoS Slowloris                     3859
DoS Slowhttptest - Attempted      3368
DoS Slowloris - Attempted         1847
DoS Slowhttptest                  1740
DoS Hulk - Attempted               581
DoS GoldenEye - Attempted           80
Heartbleed                          11
Name: count, dtype: int64


## Step 4: Basic data checks

In [4]:
print('Duplicate rows:', df.duplicated().sum())
print('Missing values:', df.isna().sum().sum())
print('Infinite numeric values:', np.isinf(df.select_dtypes(include='number')).sum().sum())

print('Columns:', len(df.columns))
print(df.dtypes.value_counts())

Duplicate rows: 6
Missing values: 0
Infinite numeric values: 0
Columns: 89
int64      62
float64    25
object      2
Name: count, dtype: int64


## Step 5: Clean the target labels

In [5]:
TARGET_COLUMN = 'Label'

# Remove the very rare Heartbleed class
df = df[df[TARGET_COLUMN] != 'Heartbleed'].copy()

# Make labels such as 'DoS - Attempted' consistent with 'DoS'
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.replace(
    ' - Attempted', '', regex=False
)

print(df[TARGET_COLUMN].value_counts())

Label
BENIGN              319119
DoS Hulk            159049
DoS GoldenEye         7647
DoS Slowloris         5706
DoS Slowhttptest      5108
Name: count, dtype: int64


## Step 6: Remove identifiers and possible leakage features

IP addresses and timestamps can make a model memorize details of the dataset instead of learning general patterns. `Attempted Category` is also removed because it may reveal information closely related to the target. Source and destination ports and constant URG-flag columns are excluded to match the original experiment.

In [6]:
columns_to_drop = [
    TARGET_COLUMN,
    'Attempted Category',
    'Src IP dec',
    'Dst IP dec',
    'Timestamp',
    'Src Port',
    'Dst Port',
    'Fwd URG Flags',
    'Bwd URG Flags',
    'URG Flag Count'
]

# Keep the target separate from the input features
X = df.drop(columns=columns_to_drop, errors='ignore')
y = df[TARGET_COLUMN]

print('Features:', X.shape)
print('Target:', y.shape)
print('Target classes:')
print(y.value_counts())

Features: (496629, 79)
Target: (496629,)
Target classes:
Label
BENIGN              319119
DoS Hulk            159049
DoS GoldenEye         7647
DoS Slowloris         5706
DoS Slowhttptest      5108
Name: count, dtype: int64


## Step 7: Remove duplicate rows

In [7]:
# Combine features and target temporarily so each full record is checked
clean_data = X.copy()
clean_data[TARGET_COLUMN] = y

print('Duplicate rows before removal:', clean_data.duplicated().sum())
clean_data = clean_data.drop_duplicates().reset_index(drop=True)

X = clean_data.drop(columns=[TARGET_COLUMN])
y = clean_data[TARGET_COLUMN]

print('Dataset after removing duplicates:', clean_data.shape)
print('Features:', X.shape)
print('Target:', y.shape)
print(y.value_counts())

Duplicate rows before removal: 18769
Dataset after removing duplicates: (477860, 80)
Features: (477860, 79)
Target: (477860,)
Label
BENIGN              300354
DoS Hulk            159048
DoS GoldenEye         7647
DoS Slowloris         5703
DoS Slowhttptest      5108
Name: count, dtype: int64


## Step 8: Split into training and testing sets

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training features:', X_train.shape)
print('Testing features:', X_test.shape)
print('Training target:', y_train.shape)
print('Testing target:', y_test.shape)

Training features: (382288, 79)
Testing features: (95572, 79)
Training target: (382288,)
Testing target: (95572,)


## Step 9: Train a Decision Tree baseline

In [9]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)

dt_predictions = decision_tree.predict(X_test)

print('Decision Tree accuracy:', accuracy_score(y_test, dt_predictions))
print('Decision Tree macro F1:', f1_score(y_test, dt_predictions, average='macro'))
print('\nClassification report:')
print(classification_report(y_test, dt_predictions, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, dt_predictions))

Decision Tree accuracy: 0.9993826643786883
Decision Tree macro F1: 0.9945994568775458

Classification report:
                  precision    recall  f1-score   support

          BENIGN       1.00      1.00      1.00     60071
   DoS GoldenEye       1.00      1.00      1.00      1529
        DoS Hulk       1.00      1.00      1.00     31810
DoS Slowhttptest       0.99      0.98      0.98      1022
   DoS Slowloris       0.99      1.00      0.99      1140

        accuracy                           1.00     95572
       macro avg       1.00      0.99      0.99     95572
    weighted avg       1.00      1.00      1.00     95572

Confusion matrix:
[[60045     3    11     9     3]
 [    1  1526     0     2     0]
 [    2     0 31808     0     0]
 [   16     3     0   997     6]
 [    2     0     0     1  1137]]


## Step 10: Find the most important features

In [10]:
importance = pd.Series(
    decision_tree.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print('Top 10 features:')
display(importance.head(10).to_frame('Importance'))

Top 10 features:


,Importance
Bwd Packet Length Std,0.840793
Bwd IAT Max,0.055847
Bwd Packets/s,0.033245
Fwd Seg Size Min,0.015403
Active Min,0.014280
Total Fwd Packet,0.004724
Bwd Packet Length Mean,0.003785
Total TCP Flow Time,0.003573
FIN Flag Count,0.003338
Flow IAT Min,0.002675


## Step 11: Train and evaluate a Random Forest

In [11]:
random_forest = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
random_forest.fit(X_train, y_train)

rf_predictions = random_forest.predict(X_test)

print('Random Forest accuracy:', accuracy_score(y_test, rf_predictions))
print('Random Forest macro F1:', f1_score(y_test, rf_predictions, average='macro'))

Random Forest accuracy: 0.99976980705646
Random Forest macro F1: 0.9973755476905115


## Step 12: Test the model using the top 10 features

In [12]:
top10 = importance.head(10).index.tolist()

rf_top10 = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
rf_top10.fit(X_train[top10], y_train)

top10_predictions = rf_top10.predict(X_test[top10])

print('Top-10 Random Forest accuracy:', accuracy_score(y_test, top10_predictions))
print('Top-10 Random Forest macro F1:', f1_score(y_test, top10_predictions, average='macro'))
print('\nClassification report:')
print(classification_report(y_test, top10_predictions, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, top10_predictions))

Top-10 Random Forest accuracy: 0.999560540744151
Top-10 Random Forest macro F1: 0.9953051548527547

Classification report:
                  precision    recall  f1-score   support

          BENIGN       1.00      1.00      1.00     60071
   DoS GoldenEye       1.00      1.00      1.00      1529
        DoS Hulk       1.00      1.00      1.00     31810
DoS Slowhttptest       0.98      0.99      0.98      1022
   DoS Slowloris       0.99      0.99      0.99      1140

        accuracy                           1.00     95572
       macro avg       0.99      1.00      1.00     95572
    weighted avg       1.00      1.00      1.00     95572

Confusion matrix:
[[60051     0     5    15     0]
 [    2  1526     0     1     0]
 [    1     0 31809     0     0]
 [    3     0     0  1010     9]
 [    2     0     0     4  1134]]


## Step 13: Save the final model and feature list

In [13]:
# Save the model and the exact feature order needed by the dashboard
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

MODEL_PATH = os.path.join(OUTPUT_FOLDER, 'model1_network_attack.joblib')
FEATURES_PATH = os.path.join(OUTPUT_FOLDER, 'model1_features.joblib')
METRICS_PATH = os.path.join(OUTPUT_FOLDER, 'model1_evaluation.csv')

joblib.dump(rf_top10, MODEL_PATH, compress=3)
joblib.dump(top10, FEATURES_PATH)

metrics = pd.DataFrame([{
    'model': 'Random Forest (top 10 features)',
    'accuracy': accuracy_score(y_test, top10_predictions),
    'macro_f1': f1_score(y_test, top10_predictions, average='macro')
}])
metrics.to_csv(METRICS_PATH, index=False)

print('Saved model:', MODEL_PATH)
print('Saved feature list:', FEATURES_PATH)
print('Saved evaluation:', METRICS_PATH)
print('Features required by the model:', top10)
print('Model file size:', round(os.path.getsize(MODEL_PATH) / 1_000_000, 2), 'MB')

Saved model: /content/drive/MyDrive/ML_Project/model1_network_attack.joblib
Saved feature list: /content/drive/MyDrive/ML_Project/model1_features.joblib
Saved evaluation: /content/drive/MyDrive/ML_Project/model1_evaluation.csv
Features required by the model: ['Bwd Packet Length Std', 'Bwd IAT Max', 'Bwd Packets/s', 'Fwd Seg Size Min', 'Active Min', 'Total Fwd Packet', 'Bwd Packet Length Mean', 'Total TCP Flow Time', 'FIN Flag Count', 'Flow IAT Min']
Model file size: 1.1 MB


## Step 14: Try sample predictions

In [14]:
# Show a few test examples and compare predictions with the true labels
sample = X_test[top10].sample(min(10, len(X_test)), random_state=1)
sample_predictions = rf_top10.predict(sample)

results = pd.DataFrame({
    'Actual label': y_test.loc[sample.index],
    'Predicted label': sample_predictions
}, index=sample.index)

display(results)

,Actual label,Predicted label
97574,DoS Hulk,DoS Hulk
167048,DoS Hulk,DoS Hulk
370227,BENIGN,BENIGN
45345,DoS Hulk,DoS Hulk
53225,DoS Hulk,DoS Hulk
51615,DoS Hulk,DoS Hulk
438000,BENIGN,BENIGN
362280,BENIGN,BENIGN
288838,BENIGN,BENIGN
324776,BENIGN,BENIGN


## Step 15: Save one example per traffic type (for the web page buttons)

In [15]:
# one real test example for each traffic type, for the web page's example buttons
examples = X_test[top10].copy()
examples['Label'] = y_test
examples = examples.groupby('Label').head(1)

examples.to_csv(OUTPUT_FOLDER + 'model1_examples.csv', index=False)
print(examples)

        Bwd Packet Length Std  Bwd IAT Max  Bwd Packets/s  Fwd Seg Size Min  \
129990            1754.831473        60775      67.415730                20   
208307               0.000000            0      16.407160                 8   
32416                0.000000            0       0.000000                40   
352050            5816.000000      5002645       0.520919                32   
27837                0.000000            0       0.000000                40   

        Active Min  Total Fwd Packet  Bwd Packet Length Mean  \
129990           0                10                  1932.5   
208307           0                 1                   157.0   
32416      7005972                 7                     0.0   
352050           0                 9                  2908.0   
27837            0                 3                     0.0   

        Total TCP Flow Time  FIN Flag Count  Flow IAT Min             Label  
129990                89000               1             0     

# Part 2: Regression - Predict Connection Duration

**Problem:** a connection that stays open much longer than normal can be a warning sign (for example slow DoS attacks). This model learns what duration is *normal* for traffic with certain packet and byte measurements.

**Type:** Linear Regression (it predicts a number, so it is evaluated with **R²**, not accuracy).

## Step 16: Look at the target (`Flow Duration`)

It is measured in **microseconds** (max 120,000,000 = 120 seconds). It is very skewed: the median is about 0.16 seconds but the mean is about 12.6 seconds, because a few connections last very long. A straight line fits this badly, so we will use the **log** of the duration.

In [16]:
print(X_train['Flow Duration'].describe())

count    3.822880e+05
mean     1.258480e+07
std      3.128882e+07
min      1.000000e+00
25%      3.107700e+04
50%      1.590000e+05
75%      1.902153e+06
max      1.200000e+08
Name: Flow Duration, dtype: float64


## Step 17: Choose safe inputs and take the log

**Columns we must NOT use (data leakage):** anything with `IAT`, `Active`, `Idle`, `/s` (per second), `Bulk Rate` and `Total TCP Flow Time`. They are time-based, so they contain the duration itself and would give a fake high R².

We also dropped `Subflow Fwd/Bwd Packets`: they are 0 in almost every row, so they carry no information.

We use 8 packet-count and packet-size columns. `log1p` means log(1 + x). `clip(lower=0)` stops negative values from breaking the log.

In [17]:
reg_features = ['Average Packet Size', 'Total Fwd Packet',
                'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
                'Total Length of Bwd Packet', 'Total Bwd packets',
                'Total Length of Fwd Packet', 'Packet Length Max']

# log squashes the huge values so a straight line fits better
Xr_train = np.log1p(X_train[reg_features].clip(lower=0))
Xr_test = np.log1p(X_test[reg_features].clip(lower=0))

yr_train = np.log1p(X_train['Flow Duration'])
yr_test = np.log1p(X_test['Flow Duration'])

## Step 18: Train Linear Regression and report R²

In [18]:
reg = LinearRegression()
reg.fit(Xr_train, yr_train)

pred = reg.predict(Xr_test)
print('R2 (log scale):', r2_score(yr_test, pred))

R2 (log scale): 0.5267404155649837


 Our R² is about **0.53**, so the model explains a bit over half of the variation in connection duration. The rest depends on things this dataset does not contain.

(We report R² on the log scale. Converting back to seconds before scoring gives a meaningless huge negative number, because a small log error becomes a huge error in seconds.)

## Step 19: Save the regression model, its inputs, and 3 examples (short, medium, long)

In [19]:
joblib.dump(reg, OUTPUT_FOLDER + 'model4_duration_regression.joblib')
joblib.dump(reg_features, OUTPUT_FOLDER + 'model4_features.joblib')

ex = X_test[reg_features + ['Flow Duration']].sort_values('Flow Duration')
n = len(ex)
ex = ex.iloc[[n // 10, n // 2, n * 9 // 10]]
ex.to_csv(OUTPUT_FOLDER + 'model4_examples.csv', index=False)
print(ex)

        Average Packet Size  Total Fwd Packet  Fwd Packet Length Mean  \
315880            78.500000                 2               47.000000   
176922           854.785714                 8               46.500000   
458410           125.282051                19               56.421053   

        Bwd Packet Length Mean  Total Length of Bwd Packet  Total Bwd packets  \
315880                   110.0                         220                  2   
176922                  1932.5                       11595                  6   
458410                   190.7                        3814                 20   

        Total Length of Fwd Packet  Packet Length Max  Flow Duration  
315880                          94                110            249  
176922                         372               7240         158988  
458410                        1072               1582       61258800  




**Part 1 (classification)**
- **Features (`X`)** are the network traffic measurements given to the model.
- **Target (`y`)** is the `Label` column, which tells the model which traffic class each row belongs to.
- **Data leakage** happens when a feature gives away the answer or information that would not legitimately be available when making a real prediction.
- **Train/test split:** 80% is used for training and 20% for testing.
- **Decision Tree:** a baseline model that learns decision rules from the features.
- **Random Forest:** combines multiple decision trees to make predictions.
- **Feature importance:** estimates which input features contributed most to the tree-based model's decisions.
- **Macro F1:** calculates F1 for each class and averages them equally, which is useful when class sizes differ.
- The saved model expects the **same 10 features in the same names/order** at prediction time.

**Part 2 (regression)**
- **Linear Regression** fits the best straight line from the inputs to a number; here the number is the connection duration.
- **R²** is the share of the variation in the target that the model explains (1 = perfect, 0 = no better than the average). Ours is about 0.53.
- **Why log:** durations are very skewed, so we predict log(duration) from log(inputs). R² went from 0.23 to 0.41 just by logging the inputs.
- **Why some columns were excluded:** per-second and time-gap columns contain the duration, which is leakage and would give a fake high R².
- **Limitation:** the inputs are correlated with each other (average packet size and packet length means move together), so individual coefficients should not be interpreted on their own.
